In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.animation import FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from IPython.display import HTML, display
from google.colab import files

# =====================================================
# MOVIMIENTO DE UNA CARGA EN UN CAMPO ELECTRICO 3D
# METODO DE EULER + ANIMACION
# =====================================================

# -----------------------------
# 1. Cargas fijas
# Cada carga: [x0, y0, z0, q]
# -----------------------------
cargas = np.array([
    [0.0, 0.0, 0.0,  1.0],
    [2.0, 0.0, 0.0, -1.0]
], dtype=float)

# -----------------------------
# 2. Parametros de la particula movil
# -----------------------------
q_test = 1.0
m = 1.0

# posicion y velocidad inicial
r0 = np.array([-3.0, 1.0, 0.5], dtype=float)
v0 = np.array([1.0, 0.0, 0.0], dtype=float)

# parametros numericos
dt = 0.01
N = 500

# -----------------------------
# 3. Campo electrico en un punto
# -----------------------------
def campo_en_punto(r, cargas):
    Ex = 0.0
    Ey = 0.0
    Ez = 0.0

    x, y, z = r

    for carga in cargas:
        x0, y0, z0, q = carga

        dx = x - x0
        dy = y - y0
        dz = z - z0

        r2 = dx**2 + dy**2 + dz**2
        dist = np.sqrt(r2)

        if dist < 1e-6:
            continue

        factor = q / dist**3

        Ex += factor * dx
        Ey += factor * dy
        Ez += factor * dz

    return np.array([Ex, Ey, Ez], dtype=float)

# -----------------------------
# 4. Integracion con Euler
# -----------------------------
trayectoria = np.zeros((N, 3))
velocidades = np.zeros((N, 3))
tiempo = np.arange(0, N * dt, dt)

r = r0.copy()
v = v0.copy()

for n in range(N):
    trayectoria[n] = r
    velocidades[n] = v

    E = campo_en_punto(r, cargas)

    v = v + (q_test / m) * E * dt
    r = r + v * dt

# -----------------------------
# 5. Malla para dibujar algunos vectores de campo
# -----------------------------
grid = np.linspace(-3, 3, 5)

Xf, Yf, Zf = [], [], []
Uf, Vf, Wf = [], [], []

for x in grid:
    for y in grid:
        for z in grid:
            punto = np.array([x, y, z], dtype=float)
            E = campo_en_punto(punto, cargas)

            # normalizar para que las flechas no queden gigantes
            norma = np.linalg.norm(E)
            if norma > 1e-8:
                E = E / norma

            Xf.append(x)
            Yf.append(y)
            Zf.append(z)
            Uf.append(E[0])
            Vf.append(E[1])
            Wf.append(E[2])

Xf = np.array(Xf)
Yf = np.array(Yf)
Zf = np.array(Zf)
Uf = np.array(Uf)
Vf = np.array(Vf)
Wf = np.array(Wf)

# -----------------------------
# 6. Figura y ejes
# -----------------------------
fig = plt.figure(figsize=(10, 5))
ax3d = fig.add_subplot(121, projection='3d', facecolor='whitesmoke')
axv  = fig.add_subplot(122, facecolor='whitesmoke')

# limites de la figura
lim = 4.0
ax3d.set_xlim([-lim, lim])
ax3d.set_ylim([-lim, lim])
ax3d.set_zlim([-lim, lim])

ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.set_title("Trayectoria en un campo electrico 3D")
ax3d.view_init(elev=25, azim=45)

# dibujar campo estatico
ax3d.quiver(Xf, Yf, Zf, Uf, Vf, Wf,
            length=0.45, normalize=False,
            color='gray', alpha=0.55)

# dibujar cargas fijas
for carga in cargas:
    x0, y0, z0, q = carga
    if q > 0:
        ax3d.scatter(x0, y0, z0, s=180, color='red')
    else:
        ax3d.scatter(x0, y0, z0, s=180, color='blue', marker='s')

# grafica de velocidad
rapidez = np.linalg.norm(velocidades, axis=1)
axv.set_xlim([0, tiempo[-1]])
axv.set_ylim([0, max(rapidez) * 1.1 + 1e-6])
axv.set_xlabel("t (s)")
axv.set_ylabel("Rapidez")
axv.set_title("Rapidez de la particula")

# -----------------------------
# 7. Elementos graficos animados
# -----------------------------
linea_tray, = ax3d.plot([], [], [], 'k-', lw=2)
punto_mov,  = ax3d.plot([], [], [], 'ko', ms=6)
linea_v,    = axv.plot([], [], 'r-', lw=2)

trace_x, trace_y, trace_z = [], [], []

# -----------------------------
# 8. Funciones de animacion
# -----------------------------
def init():
    linea_tray.set_data_3d([], [], [])
    punto_mov.set_data_3d([], [], [])
    linea_v.set_data([], [])
    return linea_tray, punto_mov, linea_v

def update(i):
    x = trayectoria[i, 0]
    y = trayectoria[i, 1]
    z = trayectoria[i, 2]

    trace_x.append(x)
    trace_y.append(y)
    trace_z.append(z)

    linea_tray.set_data_3d(trace_x, trace_y, trace_z)
    punto_mov.set_data_3d([x], [y], [z])

    linea_v.set_data(tiempo[:i+1], rapidez[:i+1])

    return linea_tray, punto_mov, linea_v

# usar menos cuadros para que no sea tan pesado
step = 4
frames = range(0, len(tiempo), step)

ani = FuncAnimation(fig, update, frames=frames,
                    init_func=init, interval=30, blit=True)

plt.close(fig)

# -----------------------------
# 9. Mostrar animacion en notebook / Colab
# -----------------------------
display(HTML(ani.to_jshtml()))

# -----------------------------
# 10. Guardar GIF o MP4 y descargar
# -----------------------------

# En mp4
writer = FFMpegWriter(fps=20)
nombre = "Trayectoria_carga_3D_animada.mp4"
ani.save(nombre,
         writer=writer)

# En gif
#nombre = "Trayectoria_carga_3D_animada.gif"
#ani.save(nombre, writer=PillowWriter(fps=20))

files.download(nombre)

Output hidden; open in https://colab.research.google.com to view.